# **Phase 1 : Data Loading**

In [22]:
import pandas as pd
import nltk
from nltk import word_tokenize, WordNetLemmatizer , pos_tag , word_tokenize
from nltk.corpus import wordnet
from nltk.corpus import stopwords
import string
nltk.download("stopwords")
import re

[nltk_data] Downloading package stopwords to C:\Users\Abdallah
[nltk_data]     Elbaz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
data= pd.read_csv('../data/sentiment_data.csv')
data.head()


,Unnamed: 0,Comment,Sentiment
0,0,lets forget apple pay required brand new iphon...,1
1,1,nz retailers don’t even contactless credit car...,0
2,2,forever acknowledge channel help lessons ideas...,2
3,3,whenever go place doesn’t take apple pay doesn...,0
4,4,apple pay convenient secure easy use used kore...,2


In [24]:
data['Sentiment'].value_counts()

Sentiment
2    103059
1     82972
0     55114
Name: count, dtype: int64

In [25]:
data.duplicated().sum()

np.int64(0)

In [26]:
data.isnull().sum()

Unnamed: 0      0
Comment       217
Sentiment       0
dtype: int64

In [27]:
data=data.dropna()

In [28]:
data_z=data[data['Sentiment']==0]
data_o=data[data['Sentiment']==1]
data_t=data[data['Sentiment']==2]

In [29]:
full_data=pd.concat([data_z, data_o[:len(data_z)], data_t[:len(data_z)]])

In [30]:
full_data=full_data.sample(frac=1, random_state=42)

# **Phase 2 : Data preprocessing**

In [31]:

def get_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    elif tag.startswith('N'):
        return wordnet.NOUN
    return wordnet.NOUN

def preprocessing_text(text):
    punctuations = set(string.punctuation)
    StopWords = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    new_tokens = [ lemmatizer.lemmatize(word.lower(),get_pos(tag)) for word,tag in tags 
                      if ( word not in StopWords) and ( word not in punctuations ) and (word.isdigit() == 0)]
        
    processed_sent = " ".join(new_tokens)
    return processed_sent
        

In [32]:
full_data['clean']=full_data['Comment'].apply(preprocessing_text)


In [33]:
data.to_csv("../data/english_cleaned.csv")

In [36]:
from collections import Counter
count={}
for cls, group in full_data.groupby('Sentiment'):
    text = ' '.join(group['clean'])
    count[cls]=Counter(text.split())

for class_name, counts in count.items():
    print(f"\nClass {class_name}:")
    print(counts)


Class 0:
Counter({'modi': 33900, 'india': 6168, 'get': 5480, 'like': 5225, 'people': 4838, 'say': 4744, 'make': 3991, 'go': 3686, 'dont': 3680, 'give': 3632, 'bjp': 3495, 'one': 3474, 'congress': 3367, 'time': 3349, 'work': 3201, 'know': 3068, 'poor': 2982, 'take': 2948, 'year': 2945, 'even': 2825, 'im': 2659, 'want': 2656, 'vote': 2588, 'hate': 2537, 'govt': 2536, 'think': 2498, 'see': 2462, 'come': 2446, 'bad': 2428, 'election': 2391, 'indian': 2270, 'money': 2238, 'day': 2217, 'country': 2134, 'use': 2101, 'cant': 2064, 'would': 2056, 'need': 1977, 'back': 1942, 'also': 1901, 'narendra': 1888, 'app': 1874, 'rahul': 1813, 'party': 1755, 'modis': 1724, 'fail': 1675, 'fake': 1675, 'try': 1612, 'still': 1609, 'thing': 1578, 'show': 1575, 'never': 1562, 'well': 1561, 'miss': 1555, 'government': 1528, 'every': 1513, 'job': 1471, 'much': 1463, 'gandhi': 1458, 'promise': 1449, 'today': 1431, 'power': 1425, 'ask': 1423, 'support': 1401, 'last': 1400, 'feel': 1393, 'nation': 1392, 'really': 

# **Phase 3 : Convert to Numerical Data**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
cv = TfidfVectorizer(max_features=10000)
x = cv.fit_transform(full_data['clean']).toarray().astype('float32')


In [40]:
y = full_data['Sentiment']

# **Phase 4 : train_test_split**

In [41]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(x,y , train_size=0.8)

# **Phase 5 : Model Selection**

In [42]:
from sklearn.naive_bayes import GaussianNB


model = GaussianNB()
model.fit(X_train,y_train)

GaussianNB()

# **Phase 6 : Model Evaluation**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))

print(confusion_matrix(y_test,y_pred))

# **Phase 7 : Save Model**

In [44]:
import pickle

with open('../models/model_english.pkl', 'wb') as file:
    pickle.dump(model, file)

In [45]:
with open('../models/vectorizer_english.pkl', 'wb') as file:
    pickle.dump(cv, file)